# Financial Index Forecasting — Reproducible Colab Research Pipeline
**Companion implementation for:** *An Applied Comparison of LSTM, Gradient Boosting, and ARIMA for Financial Forecasting*

**Models:** ARIMA (univariate), gradient boosting, and two-layer LSTM  
**Indices:** S&P 500 (`^GSPC`), Dow Jones (`^DJI`), NASDAQ Composite (`^IXIC`)  
**Targets:** future closing **index levels** at 1, 5, 10, 20 trading sessions  
**Splits:** train 2013–2019 · validate 2020–2021 · test 2022–2023


## 1 · Dependencies and execution configuration
Enable **Runtime → Change runtime type → T4 GPU** for LSTM. A full grid across all indices, horizons, predictor groups and seeds entails many fits and may require substantial Colab compute. Code exports all predictions, metrics and settings rather than relying on screenshot evidence.

In [ ]:
%pip -q install "yfinance>=0.2.40" "statsmodels>=0.14.2" "scikit-learn>=1.4.2" "torch>=2.2" "transformers>=4.42,<5" "pandas>=2.2" "matplotlib>=3.8" "seaborn>=0.13" "pyarrow>=15" "tqdm>=4.66" "huggingface_hub>=0.23"

In [ ]:
import os, sys, json, random, hashlib, warnings, platform, itertools, time
from pathlib import Path
from datetime import datetime
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import acf
import yfinance as yf
from tqdm.auto import tqdm

RUN_MODE = 'quick'  # 'quick' = smoke-test configuration; 'full' = manuscript grids
INPUT_MODE = 'auto' # auto: use available sources; 'strict_full' requires macro+Reuters
RUN_ARIMA = True
RUN_GBM = True
RUN_LSTM = True
RUN_FEATURE_ABLATIONS = False # set True for market, market+macro, market+sentiment, all
RUN_HORIZONS = [1, 5, 10, 20] if RUN_MODE == 'full' else [1]
RUN_INDICES = ['^GSPC','^DJI','^IXIC'] if RUN_MODE == 'full' else ['^GSPC']
SEEDS = [21,42,84] if RUN_MODE == 'full' else [42]
LOOKBACK, BATCH_SIZE, MAX_EPOCHS = 60,32,(100 if RUN_MODE=='full' else 8)
PATIENCE, MIN_DELTA = 10, 1e-5
START, END_EXCLUSIVE = '2013-01-02','2023-12-30'
TRAIN_END, VAL_END = pd.Timestamp('2019-12-31'), pd.Timestamp('2021-12-31')
OUT = Path('/content/financial_forecasting_outputs'); OUT.mkdir(parents=True, exist_ok=True)
INPUTS = Path('/content/financial_inputs'); INPUTS.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sns.set_theme(style='whitegrid', context='paper', font_scale=1.2)
print('Device:',DEVICE,'| Run:',RUN_MODE,'| Inputs:',INPUTS)
print('Runtime:',sys.version.split()[0],'PyTorch',torch.__version__)

## 2 · Input requirements and source transparency
**Public-market-only mode** can run without uploads. To enable full predictor groups, upload **both** licensed headline records and archived point-in-time macro data into `/content/financial_inputs/` (left sidebar → Files → Upload).

**`reuters_headlines.csv`** required columns: `published_at` (timezone-aware ISO timestamp), `document_id`, `source`, `headline`, `language`. Only original Reuters records to which you have legal access should be used. 

**`alfred_vintages.csv`** required columns: `series_id` (`FEDFUNDS`, `CPIAUCSL`, `UNRATE`, `INDPRO`), `date` (reference month), `vintage_date` (date of historically available vintage), `value`; optional `release_timestamp` with timezone. An ALFRED export should include **all needed vintage observations**, not merely today's latest revised series. If only vintage dates are known, the notebook conservatively admits them beginning the next trading session. This does not replace exact release-time evidence.

**Optional `market_closes.csv`** columns `Date,^GSPC,^DJI,^IXIC`: allows you to use the actual frozen market extract used in the study rather than Yahoo's later download.

**`ProsusAI/finbert`** is used for sentiment when headlines exist. Network availability and model licensing apply.

In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()

source_manifest=[]
def register_source(name, path):
    if path and Path(path).exists():
        source_manifest.append({'source':name,'path':str(path),'sha256':sha256_file(path)})

def load_market():
    saved=INPUTS/'market_closes.csv'
    if saved.exists():
        frame=pd.read_csv(saved,parse_dates=['Date']).set_index('Date')
        register_source('market_user_snapshot',saved)
    else:
        frames=[]
        for ticker in RUN_INDICES:
            data=yf.download(ticker,start=START,end=END_EXCLUSIVE,auto_adjust=False,
                             progress=False,threads=False)
            if data.empty: raise RuntimeError(f'No Yahoo data for {ticker}; upload market_closes.csv')
            close=data['Close']
            if isinstance(close,pd.DataFrame): close=close.iloc[:,0]
            frames.append(close.rename(ticker))
        frame=pd.concat(frames,axis=1,join='inner')
        source_manifest.append({'source':'Yahoo Finance download','retrieved_utc':pd.Timestamp.now(tz='UTC').isoformat(),
                                'note':'not necessarily identical to 2026-10-01 manuscript snapshot'})
    frame.index=pd.to_datetime(frame.index).tz_localize(None).normalize()
    frame=frame.sort_index().loc[START:'2023-12-29',RUN_INDICES].astype(float)
    if frame.index.has_duplicates or frame.isna().any().any():
        raise ValueError('Duplicate dates or missing closes; inspect market extract')
    assert (frame>0).all().all()
    print('Market dates:',len(frame),'start/end:',frame.index.min(),frame.index.max())
    print('Year partitions:',{k:int(v.sum()) for k,v in {
        'train':frame.index<=TRAIN_END,
        'validation':(frame.index>TRAIN_END)&(frame.index<=VAL_END),
        'test':frame.index>VAL_END}.items()})
    return frame

market=load_market()
market.to_csv(OUT/'market_closes_used.csv',index_label='Date')

## 3 · Chronologically admissible macroeconomic vintages
This cell reconstructs a **conservative** point-in-time value by requiring an archived vintage to have become eligible by the 16:30 ET cutoff. For CPI and INDPRO, the last available observation and its 12-month comparator are computed from the same as-of vintage snapshot. Records carrying a release timestamp use that timestamp; vintage-date-only records enter on the next trading date. Missing predictor history is **not** silently filled with current revised data.

In [ ]:
MACRO_IDS=['FEDFUNDS','CPIAUCSL','UNRATE','INDPRO']
MACRO_NAMES=['fedfunds','cpi_yoy','unrate','indpro_yoy']

def session_cutoffs(dates):
    return pd.DatetimeIndex(dates).tz_localize('America/New_York')+pd.Timedelta(hours=16,minutes=30)

def load_macro_asof(dates):
    path=INPUTS/'alfred_vintages.csv'
    if not path.exists(): return None
    raw=pd.read_csv(path,dtype={'series_id':'string'})
    for c in ['series_id','date','vintage_date','value']:
        if c not in raw.columns: raise ValueError(f'Macro file missing: {c}')
    raw=raw[raw.series_id.isin(MACRO_IDS)].copy()
    raw['date']=pd.to_datetime(raw['date']).dt.to_period('M').dt.to_timestamp()
    raw['vintage_date']=pd.to_datetime(raw['vintage_date']).dt.normalize()
    raw['value']=pd.to_numeric(raw['value'],errors='coerce')
    raw=raw.dropna(subset=['value'])
    if 'release_timestamp' in raw and raw['release_timestamp'].notna().all():
        raw['eligible_at']=pd.to_datetime(raw['release_timestamp'],utc=True)
    else:
        # Date-only vintage: not eligible on date itself; next trading cutoff onward.
        raw['eligible_at']=(raw['vintage_date']+pd.Timedelta(days=1)).dt.tz_localize('UTC')
    register_source('ALFRED archived vintages',path)
    result=pd.DataFrame(index=dates,columns=MACRO_NAMES,dtype=float)
    by_series={sid:raw[raw.series_id==sid].sort_values(['eligible_at','vintage_date']) for sid in MACRO_IDS}
    for day, cutoff in zip(dates,session_cutoffs(dates)):
        utc=cutoff.tz_convert('UTC')
        for sid,name in zip(MACRO_IDS,MACRO_NAMES):
            avail=by_series[sid][by_series[sid].eligible_at<=utc]
            if avail.empty: continue
            # latest available value for each reference month at cutoff;
            # this is a time-consistent snapshot across all reference months.
            snapshot=avail.sort_values(['eligible_at','vintage_date']).drop_duplicates('date',keep='last')
            snapshot=snapshot.set_index('date')['value'].sort_index()
            latest=snapshot.index.max(); value=float(snapshot.loc[latest])
            if sid in ['CPIAUCSL','INDPRO']:
                previous=latest-pd.DateOffset(months=12)
                if previous not in snapshot.index or float(snapshot.loc[previous])==0: continue
                value=100*(value/float(snapshot.loc[previous])-1)
            result.at[day,name]=value
    # Deliberately fail on missing required availability; do not backfill.
    if result.isna().any().any():
        print('Macro incomplete: missing by column',result.isna().sum().to_dict())
        return None
    return result

macro=load_macro_asof(market.index)
print('Macroeconomic input:', 'available' if macro is not None else 'not provided / incomplete')

## 4 · Reuters headline filtering and FinBERT sentiment
Each eligible headline is assigned to the **first trading-session cutoff at or after** its publication. Relevance filtering, within-interval normalized deduplication, FinBERT inference and interval aggregation follow the manuscript. Historical inference with a retrospectively chosen FinBERT checkpoint **does not imply historical availability of this checkpoint**.

In [ ]:
import re, unicodedata
RELEVANCE=re.compile(r'S&P 500|Dow Jones|Nasdaq|Wall Street|U\.S\. stocks|US stocks|Federal Reserve',re.I)
FINBERT_REV='4556d13015211d73dccd3fdd39d39232506f3e43'

def clean_headline(s):
    return ' '.join(unicodedata.normalize('NFKC',str(s)).lower().split())

def load_sentiment(dates):
    path=INPUTS/'reuters_headlines.csv'
    if not path.exists(): return None
    raw=pd.read_csv(path)
    req=['published_at','document_id','source','headline','language']
    if any(k not in raw for k in req): raise ValueError(f'Headline file must contain {req}')
    raw=raw.dropna(subset=req).copy()
    raw['published_at']=pd.to_datetime(raw['published_at'],utc=True,errors='coerce')
    raw=raw.dropna(subset=['published_at'])
    raw=raw[raw.language.astype(str).str.lower().isin(['en','eng','english'])]
    raw=raw[raw.headline.astype(str).str.contains(RELEVANCE,na=False)]
    raw=raw.sort_values(['published_at','document_id']).copy()
    raw['normalized']=raw.headline.map(clean_headline)
    cutoffs=session_cutoffs(dates).tz_convert('UTC')
    # searchsorted right-side: headline exactly at cutoff is included that day
    assignment=cutoffs.searchsorted(pd.DatetimeIndex(raw.published_at),side='left')
    valid=assignment<len(dates)
    raw=raw.loc[valid].copy(); raw['session_idx']=assignment[valid]
    raw=raw.drop_duplicates(['session_idx','normalized'],keep='first')
    if raw.empty: raise ValueError('No eligible Reuters headlines after filters')
    register_source('licensed Reuters headlines',path)
    from transformers import AutoTokenizer,AutoModelForSequenceClassification
    tok=AutoTokenizer.from_pretrained('ProsusAI/finbert',revision=FINBERT_REV)
    model=AutoModelForSequenceClassification.from_pretrained('ProsusAI/finbert',revision=FINBERT_REV).to(DEVICE)
    model.eval(); labels={str(v).lower():int(k) for k,v in model.config.id2label.items()}
    if 'positive' not in labels or 'negative' not in labels: raise ValueError(f'Unexpected labels: {labels}')
    scores=[]
    with torch.inference_mode():
        for start in tqdm(range(0,len(raw),32),desc='FinBERT'):
            batch=raw.headline.iloc[start:start+32].astype(str).tolist()
            inputs=tok(batch,truncation=True,padding=True,max_length=128,return_tensors='pt').to(DEVICE)
            prob=torch.softmax(model(**inputs).logits,dim=-1).cpu().numpy()
            scores.extend((prob[:,labels['positive']]-prob[:,labels['negative']]).tolist())
    raw['sentiment_score']=scores
    agg=raw.groupby('session_idx').sentiment_score.agg(['mean','count'])
    result=pd.DataFrame(0.,index=dates,columns=['sentiment_mean','sentiment_log_count'])
    result.iloc[agg.index,0]=agg['mean'].to_numpy()
    result.iloc[agg.index,1]=np.log1p(agg['count'].to_numpy())
    print('Retained headlines:',len(raw))
    return result

sentiment=load_sentiment(market.index)
print('Sentiment input:', 'available' if sentiment is not None else 'not supplied')
if INPUT_MODE=='strict_full' and (macro is None or sentiment is None):
    raise RuntimeError('strict_full requires complete point-in-time ALFRED + licensed Reuters data')

## 5 · Market features, training-only transformations and sample eligibility
Eight market features: close; last two log returns; 5- and 20-session log returns; close/SMA5−1; close/SMA20−1; 20-session log-return volatility (`ddof=0`). All values depend only on observations through the forecast origin. Predictor scaling uses training dates **before** building overlapping windows; target scaling uses training index-level minimum and maximum.

In [ ]:
MARKET_COLS=['close','log_ret_1','log_ret_lag1','log_ret_5','log_ret_20',
             'close_sma5_ratio','close_sma20_ratio','volatility20']
SENT_COLS=['sentiment_mean','sentiment_log_count']
GROUPS={
 'market':['market'],
 'market_macro':['market','macro'],
 'market_sentiment':['market','sentiment'],
 'all':['market','macro','sentiment']
}

def market_features(close):
    p=close.astype(float); r=np.log(p/p.shift(1))
    f=pd.DataFrame(index=p.index)
    f['close']=p; f['log_ret_1']=r; f['log_ret_lag1']=r.shift(1)
    f['log_ret_5']=np.log(p/p.shift(5))
    f['log_ret_20']=np.log(p/p.shift(20))
    f['close_sma5_ratio']=p/p.rolling(5).mean()-1
    f['close_sma20_ratio']=p/p.rolling(20).mean()-1
    f['volatility20']=r.rolling(20).std(ddof=0)
    assert f.iloc[20:].notna().all().all()
    return f

def make_feature_df(ticker,group):
    chunks=[market_features(market[ticker])]
    if 'macro' in GROUPS[group]:
        if macro is None: raise ValueError('Macro input missing')
        chunks.append(macro)
    if 'sentiment' in GROUPS[group]:
        if sentiment is None: raise ValueError('Sentiment input missing')
        chunks.append(sentiment)
    f=pd.concat(chunks,axis=1)
    if f.iloc[20:].isna().any().any(): raise ValueError('Missing feature values after feature warmup')
    return f

def prepare_data(ticker, group, horizon):
    f=make_feature_df(ticker,group)
    dates=market.index; close=market[ticker].to_numpy(dtype=float)
    train_mask=(dates<=TRAIN_END)
    train_close=close[train_mask]
    low, high=float(train_close.min()),float(train_close.max())
    if high<=low: raise ValueError('Target range zero')
    z=(close-low)/(high-low)
    train_features=f.loc[train_mask].iloc[20:]
    mu=train_features.mean(axis=0); sigma=train_features.std(axis=0,ddof=0).replace(0.,1.)
    normalized=((f-mu)/sigma).to_numpy(dtype=np.float32)
    pieces={p:{'X':[],'y':[],'origins':[],'targets':[],'origin_idx':[],'target_idx':[]} for p in ['train','validation','test']}
    n=len(dates)
    for origin in range(79,n-horizon):
        target=origin+horizon
        if dates[target]<=TRAIN_END:
            partition='train'
        elif dates[target]<=VAL_END and dates[origin]>=TRAIN_END:
            partition='validation'
        elif dates[target]>VAL_END and dates[origin]>=VAL_END:
            partition='test'
        else: continue
        window=normalized[origin-LOOKBACK+1:origin+1]
        if not np.isfinite(window).all(): raise RuntimeError('Nonfinite features in eligible window')
        g=pieces[partition]
        g['X'].append(window);g['y'].append(z[target]);g['origins'].append(dates[origin]);g['targets'].append(dates[target]);g['origin_idx'].append(origin);g['target_idx'].append(target)
    for p,g in pieces.items():
        g['X']=np.asarray(g['X'],dtype=np.float32)
        g['y']=np.asarray(g['y'],dtype=np.float32)
    return {'parts':pieces,'low':low,'high':high,'columns':f.columns.tolist(),'ticker':ticker,
            'group':group,'horizon':horizon}

AVAILABLE_GROUPS=['market']
if macro is not None: AVAILABLE_GROUPS.append('market_macro')
if sentiment is not None: AVAILABLE_GROUPS.append('market_sentiment')
if macro is not None and sentiment is not None: AVAILABLE_GROUPS.append('all')
SELECTED_GROUPS=AVAILABLE_GROUPS if RUN_FEATURE_ABLATIONS else [AVAILABLE_GROUPS[-1]]
print('Feature groups:',SELECTED_GROUPS)
probe=prepare_data(RUN_INDICES[0],SELECTED_GROUPS[-1],RUN_HORIZONS[0])
print('Window:',probe['parts']['train']['X'].shape,'| feature columns:',probe['columns'])
print('Counts:',{k:len(v['y']) for k,v in probe['parts'].items()})

## 6 · Metrics, reproducibility controls, and prediction record schema

In [ ]:
def set_seed(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False

def metrics(y,pred):
    y,pred=np.asarray(y,dtype=float),np.asarray(pred,dtype=float)
    return {'MAE':float(mean_absolute_error(y,pred)),
            'RMSE':float(np.sqrt(mean_squared_error(y,pred))),
            'R2':float(r2_score(y,pred)) if np.var(y)>0 else np.nan}

pred_records=[]; selection_records=[]; fit_records=[]
def write_predictions(d,partition,model,seed,pred,config):
    part=d['parts'][partition]
    if len(pred)!=len(part['y']): raise AssertionError('Prediction/target length mismatch')
    for j,value in enumerate(pred):
        pred_records.append({'index':d['ticker'],'model':model,'predictor_set':d['group'],
                             'seed':seed,'horizon':d['horizon'],'split':partition,
                             'origin_date':str(part['origins'][j].date()),
                             'target_date':str(part['targets'][j].date()),
                             'actual_scaled':float(part['y'][j]),'predicted_scaled':float(value),
                             'actual_points':float(d['low']+(d['high']-d['low'])*part['y'][j]),
                             'predicted_points':float(d['low']+(d['high']-d['low'])*value),
                             'config':json.dumps(config,sort_keys=True)})
    return metrics(part['y'],pred)

## 7 · Gradient boosting: validation-selected hyperparameter grids
Full protocol: 18 configurations × seeds 21, 42, 84 per index × horizon × predictor group, with squared-error loss, subsample 0.8 and minimum leaf size 10. Choose configuration by mean seed-level **validation MAE**, retain all seed-specific fitted estimators. Quick mode uses one configuration and one seed, and is **not** comparable to paper metrics.

In [ ]:
GBM_GRID=([{'n_estimators':n,'learning_rate':lr,'max_depth':dep}
           for n,lr,dep in itertools.product([200,300,500],[.01,.03,.05],[2,3])]
          if RUN_MODE=='full' else [{'n_estimators':100,'learning_rate':.03,'max_depth':2}])

def fit_gbm(d):
    tr=d['parts']['train'];va=d['parts']['validation'];te=d['parts']['test']
    xtr=tr['X'].reshape(len(tr['X']),-1); xva=va['X'].reshape(len(va['X']),-1)
    xte=te['X'].reshape(len(te['X']),-1)
    best=None
    for cfg in tqdm(GBM_GRID,desc=f"GBM {d['ticker']} h{d['horizon']} {d['group']}",leave=False):
        models={};scores=[]
        for seed in SEEDS:
            model=GradientBoostingRegressor(loss='squared_error',min_samples_leaf=10,
                min_samples_split=2,subsample=.8,max_features=None,criterion='friedman_mse',
                ccp_alpha=0.,warm_start=False,n_iter_no_change=None,random_state=seed,**cfg)
            model.fit(xtr,tr['y']); score=metrics(va['y'],model.predict(xva))['MAE']
            models[seed]=model;scores.append(score)
        quality=float(np.mean(scores))
        if best is None or quality<best['score']:
            best={'score':quality,'cfg':cfg.copy(),'models':models}
    selection_records.append({'index':d['ticker'],'horizon':d['horizon'],'predictor_set':d['group'],
                              'model':'GBM','validation_MAE':best['score'],**best['cfg']})
    for seed,model in best['models'].items():
        m=write_predictions(d,'test','GBM',seed,model.predict(xte),best['cfg'])
        fit_records.append({'index':d['ticker'],'horizon':d['horizon'],'predictor_set':d['group'],
                            'model':'GBM','seed':seed,**m})

## 8 · LSTM: two recurrent layers, chronological training and validation checkpointing
Full architecture grid: 2 hidden widths × 2 dropouts × 2 learning rates = 8 configurations × 3 seeds. MSE/Adam, batch 32 (no shuffle), gradient clip 1.0, 100 epochs, patience 10, minimum improvement `1e-5`. Checkpoint uses the minimum observed validation MAE, not the final epoch. Parameters are **not** refitted on validation data.

In [ ]:
class ForecastLSTM(nn.Module):
    def __init__(self,n_features,hidden=64,dropout=.2):
        super().__init__()
        self.lstm=nn.LSTM(input_size=n_features,hidden_size=hidden,num_layers=2,
                          dropout=dropout,batch_first=True,bidirectional=False)
        self.head=nn.Linear(hidden,1)
    def forward(self,x):
        states,_=self.lstm(x)
        return self.head(states[:,-1,:]).squeeze(-1)

LSTM_GRID=([{'hidden':h,'dropout':dp,'lr':lr}
            for h,dp,lr in itertools.product([32,64],[.1,.2],[1e-3,5e-4])]
           if RUN_MODE=='full' else [{'hidden':32,'dropout':.1,'lr':1e-3}])

def make_loader(p,shuffle=False):
    ds=TensorDataset(torch.from_numpy(p['X']),torch.from_numpy(p['y']))
    return DataLoader(ds,batch_size=BATCH_SIZE,shuffle=shuffle,drop_last=False)

def lstm_predict(model,p):
    model.eval();out=[]
    with torch.inference_mode():
        for x,y in make_loader(p):out.extend(model(x.to(DEVICE)).cpu().numpy().tolist())
    return np.asarray(out)

def train_one_lstm(d,cfg,seed):
    set_seed(seed)
    model=ForecastLSTM(len(d['columns']),hidden=cfg['hidden'],dropout=cfg['dropout']).to(DEVICE)
    optim=torch.optim.Adam(model.parameters(),lr=cfg['lr'],betas=(.9,.999),eps=1e-8,weight_decay=0)
    loss_fn=nn.MSELoss();tr=d['parts']['train'];va=d['parts']['validation']
    train_loader=make_loader(tr,shuffle=False)
    best_mae=float('inf');best_state=None;best_epoch=0;wait=0
    for ep in range(MAX_EPOCHS):
        model.train()
        for x,y in train_loader:
            x,y=x.to(DEVICE),y.to(DEVICE)
            optim.zero_grad(set_to_none=True)
            loss=loss_fn(model(x),y);loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.0)
            optim.step()
        val_mae=metrics(va['y'],lstm_predict(model,va))['MAE']
        if val_mae<best_mae:
            best_mae=val_mae;best_epoch=ep+1
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        if ep==0: stop_reference=val_mae;wait=0
        elif val_mae < stop_reference-MIN_DELTA:stop_reference=val_mae;wait=0
        else:wait+=1
        if wait>=PATIENCE:break
    model.load_state_dict(best_state)
    return model,best_mae,best_epoch

def fit_lstm(d):
    chosen=None
    for cfg in tqdm(LSTM_GRID,desc=f"LSTM {d['ticker']} h{d['horizon']} {d['group']}",leave=False):
        runs={};validation_scores=[]
        for seed in SEEDS:
            model,mae,epoch=train_one_lstm(d,cfg,seed)
            runs[seed]=(model,epoch,mae); validation_scores.append(mae)
        avg=float(np.mean(validation_scores))
        if chosen is None or avg<chosen['score']:
            chosen={'score':avg,'cfg':cfg.copy(),'runs':runs}
    selection_records.append({'index':d['ticker'],'horizon':d['horizon'],'predictor_set':d['group'],
                              'model':'LSTM','validation_MAE':chosen['score'],**chosen['cfg']})
    for seed,(model,epoch,vmae) in chosen['runs'].items():
        config={**chosen['cfg'],'retained_epoch':epoch}
        m=write_predictions(d,'test','LSTM',seed,lstm_predict(model,d['parts']['test']),config)
        fit_records.append({'index':d['ticker'],'horizon':d['horizon'],'predictor_set':d['group'],
                            'model':'LSTM','seed':seed,'selected_epoch':epoch,
                            'validation_MAE':vmae,**m})
        del model
    if DEVICE.type=='cuda':torch.cuda.empty_cache()

## 9 · ARIMA: fixed coefficients and sequential state updates
ARIMA uses **only** univariate closes. Full order search `p,q ∈ {0,…,5}`, `d=1`, no trend/drift/exogenous predictors. A candidate is fit only through 2019; its state receives 2020–2023 observations sequentially using `append(refit=False)`. Horizon-specific validation MAE selects an order; no re-estimation occurs during testing. The quick-mode order grid is intentionally limited.

In [ ]:
ARIMA_ORDERS=list(itertools.product(range(6),[1],range(6))) if RUN_MODE=='full' else [(1,1,0),(0,1,1),(5,1,0)]

def arima_sequential_predict(fitted,close,origins,horizon,start_idx):
    # Candidate fit extends to the fixed training end; subsequent observations advance state.
    state=fitted
    cursor=start_idx # index of latest observed close already assimilated
    forecasts=[]
    for origin in origins:
        # append only observations that have become available since prior origin
        if origin>cursor:
            state=state.append(close[cursor+1:origin+1],refit=False)
            cursor=origin
        forecasts.append(float(np.asarray(state.forecast(steps=horizon))[-1]))
    return np.asarray(forecasts,dtype=float),state

def fit_arima_index(ticker,horizons):
    dates=market.index;close=market[ticker].to_numpy(dtype=float)
    end_train=np.flatnonzero(dates<=TRAIN_END)[-1]
    end_val=np.flatnonzero(dates<=VAL_END)[-1]
    fits=[]
    for order in tqdm(ARIMA_ORDERS,desc=f'ARIMA {ticker}'):
        try:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                fitted=ARIMA(close[:end_train+1],order=order,trend='n',
                             enforce_stationarity=True,enforce_invertibility=True).fit(
                             method_kwargs={'maxiter':500,'method':'lbfgs'})
            ret=fitted.mle_retvals
            if not ret.get('converged',False):continue
            scores={}
            # One sequential validation rollout; pick different horizon forecasts at same origins.
            for h in horizons:
                d=prepare_data(ticker,'market',h)
                vi=np.asarray(d['parts']['validation']['origin_idx'],dtype=int)
                pred,_=arima_sequential_predict(fitted,close,vi,h,end_train)
                zp=(pred-d['low'])/(d['high']-d['low'])
                if not np.isfinite(zp).all():break
                scores[h]=metrics(d['parts']['validation']['y'],zp)['MAE']
            if len(scores)==len(horizons):fits.append((scores,order,fitted))
        except Exception as exc:
            print('ARIMA candidate failed',ticker,order,str(exc)[:100])
    if not fits: raise RuntimeError(f'No converged ARIMA model for {ticker}')
    for h in horizons:
        score,order,fitted=min(((s[h],o,f) for s,o,f in fits),key=lambda v:v[0])
        d=prepare_data(ticker,'market',h)
        ti=np.asarray(d['parts']['test']['origin_idx'],dtype=int)
        # Append all validation observations before the first test origin.
        pred,_=arima_sequential_predict(fitted,close,ti,h,end_train)
        zp=(pred-d['low'])/(d['high']-d['low'])
        selection_records.append({'index':ticker,'horizon':h,'predictor_set':'market',
                                  'model':'ARIMA','validation_MAE':score,'order':str(order)})
        m=write_predictions(d,'test','ARIMA',None,zp,{'order':list(order)})
        fit_records.append({'index':ticker,'horizon':h,'predictor_set':'market',
                            'model':'ARIMA','seed':None,**m})

## 10 · Execute all configured model families
The full job is intentionally opt-in. Runs are individually retained; validation picks hyperparameters, not a winning test seed. You can set model flags off to run in stages.

In [ ]:
started=time.perf_counter()
if RUN_ARIMA:
    for ticker in RUN_INDICES:fit_arima_index(ticker,RUN_HORIZONS)
for ticker,h,group in itertools.product(RUN_INDICES,RUN_HORIZONS,SELECTED_GROUPS):
    data=prepare_data(ticker,group,h)
    print('Task:',ticker,'h=',h,'group=',group,'n_test=',len(data['parts']['test']['y']))
    if RUN_GBM:fit_gbm(data)
    if RUN_LSTM:fit_lstm(data)
print(f'Finished in {(time.perf_counter()-started)/60:.1f} min; predictions:',len(pred_records))

## 11 · Persist research evidence, aggregate scores, and compare with persistence
First calculate MAE/RMSE/R² within every **index × seed × horizon** group, then average across seeds, then equally across indices. Do **not** calculate RMSE/R² after pooling all predictions. Persistence is evaluated on exactly the same dates as each fitted model, and the diagnostic 119-target subset is handled separately.

In [ ]:
pred=pd.DataFrame(pred_records)
if pred.empty:raise RuntimeError('No predictions generated: enable at least one model')
pred['target_date']=pd.to_datetime(pred['target_date'])
pred['origin_date']=pd.to_datetime(pred['origin_date'])
pred.to_csv(OUT/'all_test_predictions.csv',index=False)
sel=pd.DataFrame(selection_records);sel.to_csv(OUT/'selected_models.csv',index=False)
fits=pd.DataFrame(fit_records);fits.to_csv(OUT/'run_metrics.csv',index=False)

# Metrics are already per run; seed mean, then equal-index mean.
seed_level=fits.groupby(['model','predictor_set','horizon','index'],dropna=False)[['MAE','RMSE','R2']].mean().reset_index()
aggregate=seed_level.groupby(['model','predictor_set','horizon'],dropna=False)[['MAE','RMSE','R2']].mean().reset_index()
aggregate.to_csv(OUT/'aggregate_metrics.csv',index=False)
print('Aggregate test metrics (not manuscript Table 10):')
display(aggregate.round(5))

def persistence_for_predictions(df):
    base=df.copy()
    # Use actual closing level at each recorded origin, normalized by per-index TRAIN range.
    vals=[]
    for _,row in base.iterrows():
        series=market[row['index']]; train=series.loc[:TRAIN_END]
        low, high=float(train.min()),float(train.max())
        vals.append((float(series.loc[row['origin_date']])-low)/(high-low))
    base['persistence_scaled']=vals
    return base

baseline=persistence_for_predictions(pred)
matched=[]
for keys,g in baseline.groupby(['model','index','horizon','predictor_set','seed'],dropna=False):
    score_model=metrics(g.actual_scaled,g.predicted_scaled)
    score_base=metrics(g.actual_scaled,g.persistence_scaled)
    matched.append({'model':keys[0],'index':keys[1],'horizon':keys[2],
                    'predictor_set':keys[3],'seed':keys[4],
                    'model_MAE':score_model['MAE'],'persistence_MAE':score_base['MAE'],
                    'relative_MAE':score_model['MAE']/score_base['MAE'] if score_base['MAE'] else np.nan,
                    'model_RMSE':score_model['RMSE'],'persistence_RMSE':score_base['RMSE'],
                    'relative_RMSE':score_model['RMSE']/score_base['RMSE'] if score_base['RMSE'] else np.nan})
matched=pd.DataFrame(matched);matched.to_csv(OUT/'persistence_comparison_full.csv',index=False)
print('Full-period matched persistence comparison:')
display(matched.round(4))

## 12 · S&P 500 diagnostic: first 120 targets and matched 119 targets
Seed 42 is used for GBM/LSTM, and the deterministic ARIMA run is used without a seed. For the 119-target paper-style subset, record 1 supplies the preceding value for record 2; record 1 is not scored. No models are retrained.

In [ ]:
diagnostic_rows=[]; persistence_rows=[]
for model in ['ARIMA','GBM','LSTM']:
    subset=pred[(pred['model']==model)&(pred['index']=='^GSPC')&(pred['horizon']==1)]
    if model!='ARIMA':subset=subset[subset['seed']==42]
    if subset.empty: continue
    if model!='ARIMA':
        # Prefer full predictors when available
        preferred='all' if 'all' in subset.predictor_set.values else SELECTED_GROUPS[-1]
        subset=subset[subset.predictor_set==preferred]
    subset=subset.sort_values('target_date').head(120).copy()
    if len(subset)<120:print('Diagnostic requires at least 120 test targets:',model);continue
    m=metrics(subset.actual_scaled,subset.predicted_scaled)
    diagnostic_rows.append({'model':model,'n':len(subset),**m})
    sub119=subset.iloc[1:].copy()
    previous=subset.actual_scaled.iloc[:-1].to_numpy()
    model_metrics=metrics(sub119.actual_scaled,sub119.predicted_scaled)
    pers_metrics=metrics(sub119.actual_scaled,previous)
    persistence_rows.append({'model':model,'n':119,'MAE':model_metrics['MAE'],
                             'RMSE':model_metrics['RMSE'],'persistence_MAE':pers_metrics['MAE'],
                             'persistence_RMSE':pers_metrics['RMSE'],
                             'RelMAE':model_metrics['MAE']/pers_metrics['MAE'],
                             'RelRMSE':model_metrics['RMSE']/pers_metrics['RMSE']})

diag=pd.DataFrame(diagnostic_rows);pers=pd.DataFrame(persistence_rows)
diag.to_csv(OUT/'diagnostic_120_scores.csv',index=False)
pers.to_csv(OUT/'persistence_119_scores.csv',index=False)
display(diag.round(5));display(pers.round(5))

## 13 · Publication-oriented figures: training-range normalized errors, horizons, diagnostic trajectories, residuals and ACF

In [ ]:
FIG=OUT/'figures';FIG.mkdir(exist_ok=True)
if not aggregate.empty:
    fig,axes=plt.subplots(1,2,figsize=(12,4.5),constrained_layout=True)
    display_set=aggregate[aggregate.horizon==1]
    for ax,metric in zip(axes,['MAE','RMSE']):
        sns.barplot(data=display_set,x='model',y=metric,hue='predictor_set',ax=ax)
        ax.set_title(f'One-session {metric}');ax.set_ylabel('Training-range normalized error')
    fig.savefig(FIG/'principal_model_comparison.png',dpi=300,bbox_inches='tight');plt.show()
    horizonplot=aggregate.groupby(['model','horizon'])[['MAE','RMSE']].mean().reset_index()
    fig,ax=plt.subplots(figsize=(8,4.5))
    sns.lineplot(data=horizonplot,x='horizon',y='MAE',hue='model',marker='o',ax=ax)
    ax.set(title='MAE across forecast horizons',xlabel='Trading-session horizon',ylabel='Normalized MAE')
    fig.savefig(FIG/'horizon_mae.png',dpi=300,bbox_inches='tight');plt.show()

subset=pred[(pred['index']=='^GSPC')&(pred['horizon']==1)].copy()
if not subset.empty:
    subset=subset[(subset['model']=='ARIMA')|(subset['seed']==42)]
    subset=subset.sort_values('target_date').groupby(['model','predictor_set'],dropna=False).head(120)
    if not subset.empty:
        fig,ax=plt.subplots(figsize=(12,5))
        actual=subset.drop_duplicates('target_date').sort_values('target_date')
        ax.plot(actual.target_date,actual.actual_points,label='Actual',color='black',linewidth=2)
        for name,g in subset.groupby(['model','predictor_set']):
            if len(g)>=120:ax.plot(g.target_date,g.predicted_points,label=' / '.join(name),alpha=.85)
        ax.legend();ax.set(title='S&P 500: first 120 test targets',ylabel='Closing index points')
        fig.autofmt_xdate();fig.savefig(FIG/'diagnostic_120_trajectory.png',dpi=300,bbox_inches='tight');plt.show()

for model in ['ARIMA','GBM','LSTM']:
    g=subset[subset.model==model]
    if g.empty:continue
    g=g[g.predictor_set==(SELECTED_GROUPS[-1] if model!='ARIMA' else 'market')].head(120)
    resid=(g.actual_scaled-g.predicted_scaled).to_numpy()
    if len(resid)<5:continue
    fig,ax=plt.subplots(1,2,figsize=(10,3.8),constrained_layout=True)
    ax[0].hist(resid,bins=20,edgecolor='white');ax[0].axvline(0,color='black',linestyle='--')
    ax[0].set(title=f'{model}: normalized residuals',xlabel='Actual − predicted')
    lag=min(20,len(resid)-1)
    ax[1].stem(range(1,lag+1),acf(resid,nlags=lag,fft=False)[1:]);ax[1].axhline(0,color='black')
    ax[1].set(title=f'{model}: residual ACF',xlabel='Lag (sessions)',ylabel='Autocorrelation')
    fig.savefig(FIG/f'{model.lower()}_diagnostic_residuals.png',dpi=300,bbox_inches='tight');plt.show()

## 14 · Integrity checks, environment manifest and downloadable ZIP
Exports are original calculated results. Checks confirm no test targets appear in the development partition, forecast origin precedes target, model scores correspond to their own records, and index/seed aggregation does not pool observations.

In [ ]:
assert (pred.target_date>pred.origin_date).all()
assert (pred.target_date>VAL_END).all()
assert np.isfinite(pred[['actual_scaled','predicted_scaled']]).all().all()
assert not pred.duplicated(['index','model','predictor_set','seed','horizon','origin_date','target_date']).any()
for (idx,h),g in pred.groupby(['index','horizon']):
    test_n=(market.index>VAL_END).sum()-h+1
    assert g.target_date.nunique()==test_n,(idx,h,g.target_date.nunique(),test_n)

runtime={'generated_utc':pd.Timestamp.now(tz='UTC').isoformat(),
         'mode':RUN_MODE,'index_tickers':RUN_INDICES,'horizons':RUN_HORIZONS,
         'selected_feature_groups':SELECTED_GROUPS,'seeds':SEEDS,
         'lookback':LOOKBACK,'train_end':str(TRAIN_END.date()),'validation_end':str(VAL_END.date()),
         'market_count':len(market),'has_reuters':sentiment is not None,'has_alfred':macro is not None,
         'device':str(DEVICE),'python':platform.python_version(),'pandas':pd.__version__,
         'numpy':np.__version__,'torch':torch.__version__,'sources':source_manifest,
         'note':'Independent rerun; numeric results are NOT taken from manuscript tables.'}
(OUT/'run_manifest.json').write_text(json.dumps(runtime,indent=2,default=str))
from zipfile import ZipFile,ZIP_DEFLATED
archive=Path('/content/financial_forecasting_results.zip')
with ZipFile(archive,'w',ZIP_DEFLATED) as z:
    for f in OUT.rglob('*'):
        if f.is_file():z.write(f,arcname=str(f.relative_to(OUT)))
print('Saved outputs:',archive,'size (MB):',round(archive.stat().st_size/1e6,2))
from IPython.display import display
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print('Download in Colab via left-sidebar Files panel.')

## Notes for the manuscript
- **Run level:** `full` enables the manuscript's stated parameter grids and all three seeds; `quick` is a software verification only.
- **Dataset discrepancies:** The paper's October 1, 2026 Yahoo extraction, 80,000 filtered Reuters headlines and historical ALFRED vintage snapshot must be provided to attempt comparable results. This notebook does not create or invent those inputs.
- **Reproducibility:** Results depend on available archives, package releases, data-vendor revisions, GPU/CPU numerical behavior, and computational budget. The reported manuscript metrics are reference results, not notebook outputs.
- **Interpretation:** Persistence is a critical benchmark. The paper explicitly reports persistence outperforming all fitted methods on its restricted 119-target diagnostic; a fresh run can produce different numbers.
- **Extension:** For a publication, add independent code review, explicit release metadata checks, serial-dependence-aware confidence intervals, and archive the exact source snapshots with access permissions.